# Local GGUF Eval: v1 -> v2 -> v2.1

This notebook extends the local GGUF comparison into a three-stage journey.

Version definitions:
- `v1`: SOAP-only SFT baseline. Tests how far single-template training plus prompt-time schema injection can transfer.
- `v2`: multi-template SFT trained on `soap + referral_a`. Tests whether multi-template training improves transfer beyond the `v1` baseline.
- `v2.1`: `v2` plus supervised `mse` training data. Tests whether adding MSE supervision fixes the ontology gap while preserving earlier gains.

What it does:
- loads `v1_qwen3b-soap-q4_k_m.gguf`, `v2_qwen3b-multitemplate-q4_k_m.gguf`, and `v2_1_lite-qwen3b-multitemplate-q4_k_m.gguf`
- uses prompt-time schema injection for every template
- compares `soap`, `referral_a`, and `referral_b` directly across all three models
- treats `mse` as a staged comparison: `v1` and `v2` use zero-shot `mse`, while `v2.1` uses supervised in-distribution `mse`
- defaults to Apple Metal offload when available, otherwise CPU


## Comparison Rule

- `soap`, `referral_a`, and `referral_b` are directly comparable across `v1`, `v2`, and `v2.1`.
- `mse` is a staged progression, not a pure apples-to-apples zero-shot comparison.
- Use the `mse` section to judge whether adding supervised `mse` examples improves ontology learning, not whether `v2.1` retains zero-shot `mse` transfer.


In [ ]:
import importlib.metadata as metadata
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path


def run_text(command):
    try:
        result = subprocess.run(
            command, capture_output=True, text=True, check=False
        )
        text = (result.stdout or result.stderr).strip()
        return text or "<no output>"
    except Exception as exc:
        return f"<unavailable: {exc}>"


print("Python:", sys.version)
print("Platform:", platform.platform())
print("Machine:", platform.machine())
print("CPU cores:", os.cpu_count())

if sys.platform == "darwin":
    mem_bytes = run_text(["sysctl", "-n", "hw.memsize"])
    if mem_bytes.isdigit():
        print("Unified memory (GB):", round(int(mem_bytes) / (1024 ** 3), 2))
    print("\nMetal / GPU info:")
    gpu_lines = run_text(["system_profiler", "SPDisplaysDataType"]).splitlines()
    print("\n".join(gpu_lines[:20]))
elif shutil.which("nvidia-smi"):
    print("\nGPU info:")
    print(run_text(["nvidia-smi"]))

print("\nllama-cpp-python:", metadata.version("llama-cpp-python"))

from llama_cpp import Llama, llama_cpp as lib

GPU_OFFLOAD_SUPPORTED = bool(lib.llama_supports_gpu_offload())
AUTO_N_GPU_LAYERS = -1 if GPU_OFFLOAD_SUPPORTED else 0

print("GPU offload supported:", GPU_OFFLOAD_SUPPORTED)
print("Default n_gpu_layers:", AUTO_N_GPU_LAYERS)


In [ ]:
import json
import re
import time
from functools import lru_cache

from src.data_generation.templates import REGISTRY
from src.data_generation.validate import (
    _check_mse,
    _check_referral,
    _check_soap,
    _collect_spans,
)
from src.prompts import build_inference_messages


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "models").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root from current working directory")


REPO_ROOT = find_repo_root(Path.cwd())
MODELS = {
    "v1": REPO_ROOT / "models" / "v1_qwen3b-soap-q4_k_m.gguf",
    "v2": REPO_ROOT / "models" / "v2_qwen3b-multitemplate-q4_k_m.gguf",
    "v2.1": REPO_ROOT / "models" / "v2_1_lite-qwen3b-multitemplate-q4_k_m.gguf",
}
DATA_FILES = {
    "soap": REPO_ROOT / "data" / "qwen3.5_latest" / "eval_in_dist.soap.jsonl",
    "referral_a": REPO_ROOT / "data" / "qwen3.5_latest" / "eval_in_dist.referral_a.jsonl",
    "referral_b": REPO_ROOT / "data" / "qwen3.5_latest" / "eval_zero_shot.referral_b.jsonl",
    "mse_zero_shot": REPO_ROOT / "data" / "qwen3.5_latest" / "eval_zero_shot.mse.jsonl",
    "mse_in_dist": REPO_ROOT / "data" / "qwen3.5_latest" / "eval_in_dist.mse.jsonl",
}
EVAL_PLAN = {
    "v1": [
        {"template": "soap", "split": "in_dist", "file_key": "soap", "note": "direct comparison"},
        {"template": "referral_a", "split": "in_dist", "file_key": "referral_a", "note": "direct comparison"},
        {"template": "referral_b", "split": "held_out", "file_key": "referral_b", "note": "direct comparison"},
        {"template": "mse", "split": "zero_shot", "file_key": "mse_zero_shot", "note": "staged MSE baseline"},
    ],
    "v2": [
        {"template": "soap", "split": "in_dist", "file_key": "soap", "note": "direct comparison"},
        {"template": "referral_a", "split": "in_dist", "file_key": "referral_a", "note": "direct comparison"},
        {"template": "referral_b", "split": "held_out", "file_key": "referral_b", "note": "direct comparison"},
        {"template": "mse", "split": "zero_shot", "file_key": "mse_zero_shot", "note": "staged MSE baseline"},
    ],
    "v2.1": [
        {"template": "soap", "split": "in_dist", "file_key": "soap", "note": "direct comparison"},
        {"template": "referral_a", "split": "in_dist", "file_key": "referral_a", "note": "direct comparison"},
        {"template": "referral_b", "split": "held_out", "file_key": "referral_b", "note": "direct comparison"},
        {"template": "mse", "split": "in_dist", "file_key": "mse_in_dist", "note": "staged MSE improvement"},
    ],
}
MODEL_ORDER = ["v1", "v2", "v2.1"]
DIRECT_TEMPLATES = ["soap", "referral_a", "referral_b"]
TEMPLATE_ORDER = ["soap", "referral_a", "referral_b", "mse"]

N_CTX = 4096
MAX_TOKENS = 512
N_THREADS = max(1, (os.cpu_count() or 4) - 2)
N_GPU_LAYERS = AUTO_N_GPU_LAYERS  # set to 0 to force CPU
SHOW_EXAMPLES = 1

for path in [*MODELS.values(), *DATA_FILES.values()]:
    assert path.exists(), f"Missing required file: {path}"

print("Repo root:", REPO_ROOT)
print("Models:")
for name, path in MODELS.items():
    print(f"  {name}: {path.name} ({path.stat().st_size / 1e9:.2f} GB)")
print("n_ctx:", N_CTX)
print("max_tokens:", MAX_TOKENS)
print("n_threads:", N_THREADS)
print("n_gpu_layers:", N_GPU_LAYERS)


In [ ]:
def load_jsonl(path: Path):
    with path.open() as f:
        return [json.loads(line) for line in f if line.strip()]


def print_table(rows, columns=None, digits=4):
    if not rows:
        print("No rows")
        return
    if columns is None:
        columns = list(rows[0].keys())

    def fmt(value):
        if isinstance(value, float):
            return f"{value:.{digits}f}"
        return str(value)

    widths = {}
    for col in columns:
        widths[col] = max(len(col), max(len(fmt(row.get(col, ""))) for row in rows))

    header = " | ".join(col.ljust(widths[col]) for col in columns)
    divider = "-+-".join("-" * widths[col] for col in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(fmt(row.get(col, "")).ljust(widths[col]) for col in columns))


DATASETS = {key: load_jsonl(path) for key, path in DATA_FILES.items()}

plan_rows = []
for model_name in MODEL_ORDER:
    for item in EVAL_PLAN[model_name]:
        plan_rows.append(
            {
                "model": model_name,
                "template": item["template"],
                "split": item["split"],
                "rows": len(DATASETS[item["file_key"]]),
                "file": str(DATA_FILES[item["file_key"]].relative_to(REPO_ROOT)),
                "note": item["note"],
            }
        )

print_table(
    plan_rows,
    columns=["model", "template", "split", "rows", "file", "note"],
)


In [ ]:
_OBJECT_RE = re.compile(r"\{.*\}", re.DOTALL)
_FENCE_RE = re.compile(r"^```(?:json)?\s*\n?(.*?)\n?```$", re.DOTALL)

EXPECTED_TOP_KEYS = {
    "soap": {"subjective", "objective", "assessment", "plan"},
    "referral_a": {"specialty", "patient", "reason", "history", "current_meds", "request"},
    "referral_b": {"specialty", "patient", "reason", "history", "current_meds", "request"},
    "mse": {"appearance", "behaviour", "speech", "mood", "affect", "thought", "cognition", "insight"},
}
SCHEMA_CHECKERS = {
    "soap": _check_soap,
    "referral_a": _check_referral,
    "referral_b": _check_referral,
    "mse": _check_mse,
}


def unwrap_json_object(raw: str) -> str:
    raw = raw.strip()
    fence = _FENCE_RE.match(raw)
    if fence:
        raw = fence.group(1).strip()
    match = _OBJECT_RE.search(raw)
    return match.group(0) if match else raw


def parse_prediction(raw: str):
    try:
        return json.loads(unwrap_json_object(raw))
    except Exception:
        return None


def collect_content_values(node):
    values = []
    if isinstance(node, dict):
        if {"text", "evidence_span"}.issubset(node.keys()):
            values.append(node.get("text"))
        elif {"name", "evidence_span"}.issubset(node.keys()):
            values.append(node.get("name"))
        elif {"action", "evidence_span"}.issubset(node.keys()):
            values.append(node.get("action"))
        else:
            for value in node.values():
                values.extend(collect_content_values(value))
    elif isinstance(node, list):
        for item in node:
            values.extend(collect_content_values(item))
    return values


def content_token_set(node):
    tokens = set()
    for value in collect_content_values(node):
        if isinstance(value, str) and value.strip():
            tokens.update(value.lower().split())
    return tokens


def content_jaccard(pred_label, gold_label):
    pred_tokens = content_token_set(pred_label)
    gold_tokens = content_token_set(gold_label)
    if not pred_tokens and not gold_tokens:
        return 1.0
    return len(pred_tokens & gold_tokens) / max(len(pred_tokens | gold_tokens), 1)


def non_null_content_rate(label):
    values = collect_content_values(label)
    if not values:
        return 0.0
    filled = sum(1 for value in values if isinstance(value, str) and value.strip())
    return filled / len(values)


def is_all_null_prediction(label):
    return all(not (isinstance(value, str) and value.strip()) for value in collect_content_values(label))


In [ ]:
@lru_cache(maxsize=None)
def load_llm(model_name: str):
    model_path = MODELS[model_name]
    print(f"[load] {model_name}: {model_path.name}")
    t0 = time.perf_counter()
    llm = Llama(
        model_path=str(model_path),
        n_ctx=N_CTX,
        n_gpu_layers=N_GPU_LAYERS,
        n_threads=N_THREADS,
        verbose=False,
        seed=0,
    )
    print(f"[load] ready in {time.perf_counter() - t0:.1f}s")
    return llm


def generate(model_name: str, template_name: str, transcript: str) -> str:
    llm = load_llm(model_name)
    messages = build_inference_messages(template_name, transcript)
    response = llm.create_chat_completion(
        messages=messages,
        temperature=0.0,
        max_tokens=MAX_TOKENS,
    )
    return response["choices"][0]["message"]["content"].strip()


def score_prediction(template_name: str, pred_raw: str, gold_label: dict, transcript: str):
    pred_label = parse_prediction(pred_raw)
    if pred_label is None or not isinstance(pred_label, dict):
        return {
            "parse": 0,
            "top_keys": 0,
            "schema_valid": 0,
            "content_overlap": 0.0,
            "evidence_grounding": 0.0,
            "non_null_content_rate": 0.0,
            "all_null": 1,
            "pred_label": None,
        }

    schema_err = SCHEMA_CHECKERS[template_name](pred_label)
    spans = [span for span in _collect_spans(pred_label) if span]
    if not spans:
        grounding = 0.0
    else:
        grounding = sum(1 for span in spans if span in transcript) / len(spans)

    return {
        "parse": 1,
        "top_keys": int(set(pred_label.keys()) >= EXPECTED_TOP_KEYS[template_name]),
        "schema_valid": int(schema_err is None),
        "content_overlap": content_jaccard(pred_label, gold_label),
        "evidence_grounding": grounding,
        "non_null_content_rate": non_null_content_rate(pred_label),
        "all_null": int(is_all_null_prediction(pred_label)),
        "pred_label": pred_label,
    }


In [ ]:
def summarise(model_name: str, template_name: str, split_name: str, details):
    n = len(details)
    return {
        "model": model_name,
        "template": template_name,
        "split": split_name,
        "n": n,
        "json_parse_rate": sum(d["parse"] for d in details) / n,
        "top_level_keys_rate": sum(d["top_keys"] for d in details) / n,
        "schema_valid_rate": sum(d["schema_valid"] for d in details) / n,
        "content_overlap_mean": sum(d["content_overlap"] for d in details) / n,
        "evidence_grounding_rate": sum(d["evidence_grounding"] for d in details) / n,
        "non_null_content_rate_mean": sum(d["non_null_content_rate"] for d in details) / n,
        "all_null_rate": sum(d["all_null"] for d in details) / n,
        "mean_latency_ms": sum(d["latency_ms"] for d in details) / n,
    }


def evaluate_model(model_name: str):
    summaries = []
    all_details = []

    for item in EVAL_PLAN[model_name]:
        template_name = item["template"]
        split_name = item["split"]
        file_key = item["file_key"]
        rows = DATASETS[file_key]
        label_key = REGISTRY[template_name]["label_key"]
        template_details = []
        print(f"[run] {model_name} -> {template_name} ({split_name}, {len(rows)} rows)")

        for index, row in enumerate(rows, start=1):
            t0 = time.perf_counter()
            pred_raw = generate(model_name, template_name, row["transcript"])
            latency_ms = (time.perf_counter() - t0) * 1000
            scores = score_prediction(
                template_name, pred_raw, row[label_key], row["transcript"]
            )
            detail = {
                "model": model_name,
                "template": template_name,
                "split": split_name,
                "file_key": file_key,
                "index": index,
                "transcript": row["transcript"],
                "gold": row[label_key],
                "pred_raw": pred_raw,
                "pred_label": scores.pop("pred_label"),
                "latency_ms": latency_ms,
                **scores,
            }
            template_details.append(detail)
            all_details.append(detail)
            print(
                f"  [{index:>2}/{len(rows)}] {latency_ms:>7.0f} ms  "
                f"parse={detail['parse']} schema={detail['schema_valid']} "
                f"overlap={detail['content_overlap']:.2f} "
                f"ground={detail['evidence_grounding']:.2f} "
                f"all_null={detail['all_null']}"
            )

        summaries.append(summarise(model_name, template_name, split_name, template_details))

    summaries.append(summarise(model_name, "ALL", "mixed", all_details))
    return summaries, all_details


In [ ]:
v1_summaries, v1_details = evaluate_model("v1")
v2_summaries, v2_details = evaluate_model("v2")
v2_1_summaries, v2_1_details = evaluate_model("v2.1")
all_summaries = v1_summaries + v2_summaries + v2_1_summaries
all_details = v1_details + v2_details + v2_1_details

print_table(
    all_summaries,
    columns=[
        "model",
        "template",
        "split",
        "n",
        "json_parse_rate",
        "schema_valid_rate",
        "content_overlap_mean",
        "evidence_grounding_rate",
        "non_null_content_rate_mean",
        "all_null_rate",
        "mean_latency_ms",
    ],
    digits=4,
)


In [ ]:
summary_lookup = {
    (row["model"], row["template"]): row
    for row in all_summaries
    if row["template"] != "ALL"
}

direct_progression = []
for template_name in DIRECT_TEMPLATES:
    v1_row = summary_lookup[("v1", template_name)]
    v2_row = summary_lookup[("v2", template_name)]
    v2_1_row = summary_lookup[("v2.1", template_name)]
    direct_progression.append(
        {
            "template": template_name,
            "v1_overlap": v1_row["content_overlap_mean"],
            "v2_overlap": v2_row["content_overlap_mean"],
            "v2_1_overlap": v2_1_row["content_overlap_mean"],
            "delta_v2_vs_v1": v2_row["content_overlap_mean"] - v1_row["content_overlap_mean"],
            "delta_v2_1_vs_v2": v2_1_row["content_overlap_mean"] - v2_row["content_overlap_mean"],
            "v1_all_null": v1_row["all_null_rate"],
            "v2_all_null": v2_row["all_null_rate"],
            "v2_1_all_null": v2_1_row["all_null_rate"],
            "v1_ground": v1_row["evidence_grounding_rate"],
            "v2_ground": v2_row["evidence_grounding_rate"],
            "v2_1_ground": v2_1_row["evidence_grounding_rate"],
        }
    )

print("Directly comparable templates")
print_table(
    direct_progression,
    columns=[
        "template",
        "v1_overlap",
        "v2_overlap",
        "v2_1_overlap",
        "delta_v2_vs_v1",
        "delta_v2_1_vs_v2",
        "v1_all_null",
        "v2_all_null",
        "v2_1_all_null",
        "v1_ground",
        "v2_ground",
        "v2_1_ground",
    ],
    digits=4,
)

mse_progression = [
    {
        "template": "mse",
        "v1_split": summary_lookup[("v1", "mse")]["split"],
        "v2_split": summary_lookup[("v2", "mse")]["split"],
        "v2_1_split": summary_lookup[("v2.1", "mse")]["split"],
        "v1_overlap": summary_lookup[("v1", "mse")]["content_overlap_mean"],
        "v2_overlap": summary_lookup[("v2", "mse")]["content_overlap_mean"],
        "v2_1_overlap": summary_lookup[("v2.1", "mse")]["content_overlap_mean"],
        "v1_schema": summary_lookup[("v1", "mse")]["schema_valid_rate"],
        "v2_schema": summary_lookup[("v2", "mse")]["schema_valid_rate"],
        "v2_1_schema": summary_lookup[("v2.1", "mse")]["schema_valid_rate"],
        "v1_all_null": summary_lookup[("v1", "mse")]["all_null_rate"],
        "v2_all_null": summary_lookup[("v2", "mse")]["all_null_rate"],
        "v2_1_all_null": summary_lookup[("v2.1", "mse")]["all_null_rate"],
        "v1_ground": summary_lookup[("v1", "mse")]["evidence_grounding_rate"],
        "v2_ground": summary_lookup[("v2", "mse")]["evidence_grounding_rate"],
        "v2_1_ground": summary_lookup[("v2.1", "mse")]["evidence_grounding_rate"],
    }
]

print("\nStaged MSE progression")
print_table(
    mse_progression,
    columns=[
        "template",
        "v1_split",
        "v2_split",
        "v2_1_split",
        "v1_overlap",
        "v2_overlap",
        "v2_1_overlap",
        "v1_schema",
        "v2_schema",
        "v2_1_schema",
        "v1_all_null",
        "v2_all_null",
        "v2_1_all_null",
        "v1_ground",
        "v2_ground",
        "v2_1_ground",
    ],
    digits=4,
)


In [ ]:
def show_examples(details, model_name: str, template_name: str, limit: int = 1):
    shown = 0
    for detail in details:
        if detail["model"] != model_name or detail["template"] != template_name:
            continue
        print("=" * 80)
        print(
            f"MODEL: {model_name}  TEMPLATE: {template_name}  "
            f"SPLIT: {detail['split']}  INDEX: {detail['index']}"
        )
        print("TRANSCRIPT:")
        print(detail["transcript"])
        print("\nPREDICTED RAW:")
        print(detail["pred_raw"])
        print("\nGOLD:")
        print(json.dumps(detail["gold"], indent=2))
        shown += 1
        if shown >= limit:
            break


for template_name in TEMPLATE_ORDER:
    for model_name in MODEL_ORDER:
        show_examples(all_details, model_name, template_name, limit=SHOW_EXAMPLES)


## Interpretation

This notebook is best read as a three-stage progression:

- `v1`: single-template SOAP baseline with prompt-time schema injection
- `v2`: multi-template SFT, testing whether referral transfer improves beyond the `v1` baseline
- `v2.1`: multi-template SFT plus supervised `mse`, testing whether the ontology gap narrows while preserving earlier behaviour

### Directly comparable templates

For `soap`, `referral_a`, and `referral_b`, the comparison is apples-to-apples across all three models.

- `soap`: `0.3404 -> 0.5032 -> 0.5239`
- `referral_a`: `0.5037 -> 0.7400 -> 0.7044`
- `referral_b`: `0.2651 -> 0.4678 -> 0.4816`

What this means:

- `v1 -> v2` is the main jump. Multi-template SFT adds clear value beyond schema injection alone.
- `v2 -> v2.1` mostly preserves the gains. `soap` improves slightly, `referral_b` improves slightly, and `referral_a` regresses slightly but remains much stronger than `v1`.
- Evidence grounding also trends upward through the progression, especially on `soap` and `referral_b`.

### Staged MSE progression

`mse` is not a pure zero-shot comparison in this notebook.

- `v1`: zero-shot `mse`
- `v2`: zero-shot `mse`
- `v2.1`: supervised in-distribution `mse`

The metrics still tell a useful story:

- `v1 mse`: overlap `0.0546`, schema `1.0000`, all-null `0.0000`
- `v2 mse`: overlap `0.0104`, schema `0.0000`, all-null `0.6000`
- `v2.1 mse`: overlap `0.1371`, schema `1.0000`, all-null `0.0000`

Interpretation:

- `v1` produces schema-shaped guesses, but the qualitative example shows weak field semantics.
- `v2` becomes more conservative and often collapses or violates the stricter nested MSE schema.
- `v2.1` is the first version that looks meaningfully usable for MSE: better overlap, valid nested structure, no all-null collapse, and stronger qualitative domain coverage.

### Qualitative reading

The examples matter as much as the tables:

- `referral_b`: `v1` still misassigns basic fields, while `v2` and `v2.1` produce much more coherent referral structure.
- `mse`: `v1` and `v2` both struggle with domain assignment, but `v2.1` clearly fills more of the ontology with plausible field-level structure, even if some behaviour / thought / affect boundaries are still imperfect.

### Takeaway

- `v1 -> v2` supports the claim that multi-template SFT improves transfer on nearby template families beyond prompt-time schema injection alone.
- `v2 -> v2.1` supports the claim that the `mse` failure was largely an unseen-ontology problem, not a total failure of the overall approach.
- `v2.1` does not prove robust zero-shot ontology transfer, but it does show that adding the missing ontology in training improves extraction while mostly preserving prior template behaviour.

### Caveats

- `content_overlap_mean` is still a generalized token-overlap metric. It is most useful for within-template comparison, not for comparing different templates to each other.
- `evidence_grounding_rate` can remain high even when content is mapped to the wrong field.
- `mse` should be interpreted as a staged training result, not as a fair zero-shot benchmark between all three models.
